In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, KFold
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
!pip install catboost
from catboost import CatBoostClassifier
from sklearn.metrics import f1_score
from sklearn.linear_model import LogisticRegression
import numpy as np

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import pandas as pd
import os
csv_path= os.path.join(path, "Q1_data.csv") # get question 1 data!
df= pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head() # will simply print 5 first rows

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
import matplotlib.pyplot as plt
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor="green", color="pink") #will plot a pink histogram with green edges for our target (delivery time) or any other column we may pass!
  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)
  plt.show()
check_target_distribution(df,"Delivery_Time")

In [ ]:
# Task 1: Write your code here:
if "Order_ID" in df: # if it was already deleted or DNE then it will be an error to use drop columns
  df= df.drop(columns= ['Order_ID'], axis=1)
  print("Deleted Successfully!")
df.head()

In [ ]:
# Task 2: Write your code here:
def handle_missing_values(df): # to handle and see if we even have missing values and what are they
  missing_values = df.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    df.fillna(method= 'ffill', inplace= True)
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

handle_missing_values(df)


In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here:

from sklearn.preprocessing import OneHotEncoder
categorical_cols = df.select_dtypes(include=["object"]).columns #check for categorical columns
print("Categorical Columns:", list(categorical_cols))
columns= [['Weather'], ['Traffic_Level'], ['Time_of_Day'], ['Vehicle_Type']]
for col in categorical_cols:
  encoder= OneHotEncoder(sparse_output=False)
  df[col]= encoder.fit_transform(df[[col]])
df.head()



In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler
features = df.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET
scaler = StandardScaler()  # will not give us specific range like MinMax
df[features] = scaler.fit_transform(df[features])
df.head()

In [ ]:
# Task 6: Write your code here:


In [ ]:
# Task 1: Write your code here:
x= df.drop("Delivery_Time", axis=1)
y= df['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:

models = {
    "Logistic Regression": LogisticRegression(max_iter=10000),
    "Random Forest Classifier": RandomForestClassifier(n_estimators=100),
    "CatBoost Classifier": CatBoostClassifier(verbose=0)
}

models["Ensemble"] = VotingClassifier(estimators=[
        ('lr', models["CatBoost Classifier"]), ('rf', models["Logistic Regression"]), ('gnb', models["Random Forest Classifier"])], voting="soft")

for model_name, model in models.items():
    scores_accuracy = []
    scores_precision = []
    scores_recall = []
    scores_f1 = []

    # Stratified 5-Fold Cross-Validation
    skf = KFold(n_splits=5)
    for train_index, test_index in skf.split(x, y):
        # Split data into training and testing sets
        X_Train, X_Test = x.loc[train_index, :], x.loc[test_index, :]
        y_Train, y_Test = y.iloc[train_index], y.iloc[test_index]
        # Train the model
        model.fit(X_Train, y_Train)
        # Predict on the test set
        y_pred = model.predict(X_Test)

        # Calculate metrics
        scores_f1.append(f1_score(y_Test, y_pred, average='weighted'))

    # Print the results
    print(f"{model_name} F1-Score: {np.mean(scores_f1):.4f}")
    print("\n")


In [ ]:
# Task 1: Write your code here:
# Retrieve CatBoost feature importances and sort them
catboost_model = models["CatBoost Classifier"]
catboost_importance = list(zip(x.columns, catboost_model.feature_importances_))
sorted_catboost_importance = sorted(catboost_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('CatBoost Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()

In [ ]:
# Task 2: Write your code here:

In [ ]:
# Task Bonus: Write your code here: